# Saxo Institutional Dashboard (Presentation)

Read-only Databricks notebook for the first institutional page on the live Saxo stack.
No workflow, no schedule, and no production writes are performed here.


### Source tables
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_feed`
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_health_rollup`
- `pawan_quant_research_1.v4.gold_rt_saxo_dashboard_performance_rollup`
- `pawan_quant_research_1.v4.bronze_rt_regime_signal`
- `pawan_quant_research_1.v4.bronze_rt_saxo_trade_log`


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

CATALOG = "pawan_quant_research_1"
SCHEMA = "v4"
FEED_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed"
HEALTH_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup"
PERF_TABLE = f"{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup"
REGIME_SIGNAL_TABLE = f"{CATALOG}.{SCHEMA}.bronze_rt_regime_signal"
TRADE_LOG_TABLE = f"{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log"

pd.set_option("display.max_columns", 200)


def as_float(value):
    try:
        if value is None or pd.isna(value):
            return None
        return float(value)
    except Exception:
        return None


def fmt_money(value):
    v = as_float(value)
    if v is None:
        return "N/A"
    return f"${v:,.2f}"


def fmt_pct(value):
    v = as_float(value)
    if v is None:
        return "N/A"
    return f"{v * 100:,.2f}%"


def latest_row(table_name, order_col, where_clause=""):
    q = f"SELECT * FROM {table_name} {where_clause} ORDER BY {order_col} DESC LIMIT 1"
    rows = spark.sql(q).toPandas()
    if rows.empty:
        raise ValueError(f"No rows in {table_name}")
    return rows.iloc[0]


def latest_rows(table_name, order_col, limit=360):
    q = f"SELECT * FROM (SELECT * FROM {table_name} ORDER BY {order_col} DESC LIMIT {int(limit)}) t ORDER BY {order_col} ASC"
    return spark.sql(q).toPandas()


## Executive KPI strip


In [ ]:
latest_feed = latest_row(FEED_TABLE, "feed_ts")
latest_health = latest_row(HEALTH_TABLE, "rollup_ts")
latest_perf = latest_row(PERF_TABLE, "rollup_ts")

kpis = pd.DataFrame(
        [
            ("Current regime", latest_feed.get("regime_name", "N/A")),
            ("Account equity", fmt_money(latest_feed.get("account_equity"))),
            ("Current position", latest_feed.get("actual_ticker", latest_feed.get("position", "N/A"))),
            ("Alpha vs SPY", fmt_pct(latest_feed.get("alpha_pct"))),
            ("Drawdown", fmt_pct(latest_feed.get("drawdown_pct"))),
            ("Freshness (min)", fmt_pct(latest_feed.get("dashboard_data_freshness_minutes"))),
            ("Last execution action", latest_feed.get("last_execution_action", "N/A")),
            ("1h strategy", fmt_pct(latest_perf.get("strategy_return_1h"))),
            ("1h alpha", fmt_pct(latest_perf.get("alpha_1h"))),
            ("1d strategy", fmt_pct(latest_perf.get("strategy_return_1d"))),
            ("1d alpha", fmt_pct(latest_perf.get("alpha_1d"))),
        ],
        columns=["Metric", "Value"],
    )
display(kpis)


## Strategy vs SPY with regime shading


In [ ]:
history = latest_rows(FEED_TABLE, "feed_ts", limit=300)
history = history.sort_values("feed_ts").reset_index(drop=True)
history = history[pd.notna(history["account_equity"])].copy()

if history.empty:
    raise ValueError("No feed rows available for chart")

history["account_equity"] = pd.to_numeric(history["account_equity"], errors="coerce")
history["SPY_close"] = pd.to_numeric(history["SPY_close"], errors="coerce")
history["strategy_return"] = history["account_equity"] / history["account_equity"].iloc[0] - 1.0

if history["SPY_close"].dropna().empty:
    history["spy_return"] = 0.0
else:
    spy_base = history["SPY_close"].dropna().iloc[0]
    history["spy_return"] = history["SPY_close"] / spy_base - 1.0

start_ts = history["feed_ts"].min()
regimes = spark.sql(
            f"""
            SELECT timestamp AS regime_ts, COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name
            FROM {REGIME_SIGNAL_TABLE}
            WHERE timestamp >= '{start_ts}'
            ORDER BY timestamp ASC
            """
        ).toPandas().sort_values("regime_ts")

color_by_regime = {
    "Expansion": "#2E8B57",
    "Slowdown": "#F5A623",
    "Recession": "#C0392B",
}

fig, ax = plt.subplots(figsize=(16, 6))
if len(regimes) == 0:
    print("No regime transitions found for the selected history window; rendering without regime shading.")
else:
    for i, row in regimes.iterrows():
        regime_start = pd.to_datetime(row["regime_ts"])
        regime_end = (
            pd.to_datetime(regimes.iloc[i + 1]["regime_ts"])
            if i + 1 < len(regimes)
            else pd.to_datetime(history["feed_ts"].iloc[-1])
        )
        color = color_by_regime.get(row["regime_name"], "#9E9E9E")
        ax.axvspan(regime_start, regime_end, color=color, alpha=0.08)

ax.plot(history["feed_ts"], history["strategy_return"] * 100, label="Strategy")
ax.plot(history["feed_ts"], history["spy_return"] * 100, linestyle="--", label="SPY")
ax.set_title("Strategy vs SPY (live feed history)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Return %")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)
display(fig)



## Portfolio / position


In [ ]:
portfolio_cols = [
    "position",
    "target_ticker",
    "actual_ticker",
    "account_cash",
    "positions_count",
    "positions_market_value",
    "positions_unrealized_plpc",
    "equity_vs_cash_ratio",
    "regime_age_hours",
]
display(latest_feed[portfolio_cols].to_frame(name="value").reset_index(names=["field"]))

positions_json = latest_feed.get("positions_json")
if isinstance(positions_json, str) and positions_json.strip():
    try:
        displayHTML("<pre>" + json.dumps(json.loads(positions_json), indent=2) + "</pre>")
    except Exception:
        displayHTML(f"<pre>{positions_json}</pre>")


## Risk / drawdown


In [ ]:
risk_rows = [
    ("Current drawdown", fmt_pct(latest_feed.get("drawdown_pct"))),
        ("Regime drawdown", fmt_pct(latest_feed.get("regime_drawdown_pct"))),
        ("Total return", fmt_pct(latest_feed.get("total_return_pct"))),
        ("Regime return", fmt_pct(latest_feed.get("regime_return_pct"))),
        ("Alpha", fmt_pct(latest_feed.get("alpha_pct"))),
        ("Peak equity", fmt_money(latest_feed.get("peak_equity"))),
        ("Regime peak equity", fmt_money(latest_feed.get("regime_peak_equity"))),
    ]

display(pd.DataFrame(risk_rows, columns=["Metric", "Value"]))


## Freshness / execution health


In [ ]:
health_rows = [
    ("Dashboard freshness (min)", fmt_pct(latest_feed.get("dashboard_data_freshness_minutes"))),
        ("Audit lag (min)", fmt_pct(latest_health.get("audit_lag_minutes"))),
        ("Latest audit ts", latest_feed.get("latest_audit_ts", "N/A")),
        ("Execution action", latest_feed.get("last_execution_action", "N/A")),
        ("Skip reason", latest_feed.get("last_skip_reason", "N/A")),
        ("Error reason", latest_feed.get("last_error_reason", "N/A")),
        ("Skip / 24h", latest_feed.get("last_skip_count_24h", 0)),
        ("Error / 24h", latest_feed.get("last_error_count_24h", 0)),
        ("Rate limits / 24h", latest_feed.get("last_rate_limit_count_24h", 0)),
        ("Regime transitions / 24h", latest_health.get("regime_transition_count_24h", 0)),
    ]
display(pd.DataFrame(health_rows, columns=["Metric", "Value"]))


### Optional drilldown context


In [ ]:
display(
    spark.sql(
        f"""
        SELECT timestamp, COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name, position
        FROM {REGIME_SIGNAL_TABLE}
        ORDER BY timestamp DESC
        LIMIT 20
        """
    ).toPandas()
)
